# Grid UCI BNN — Aggregated Results

Loads all saved `.pt` runs (grid_boomerang, grid_sticky_boomerang, nuts, nuts_horseshoe) across splits for each UCI dataset, computes RMSE, NLL, and CRPS on the held-out test set, then produces a paper-ready table (mean ± SEM across splits).

Noise is learned by every sampler in this pipeline (see `sazz/gpu_friendly/scripts/uci_bnn_grid.py`) -- there is no fixed-noise variant to filter on, unlike the old `uci_aggregated.ipynb`.

In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import Tensor
import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

from sazz.gpu_friendly.scripts.uci_bnn_grid import (
    load_raw_datasets, make_split, build_target, BNNConfig, BASE_SEED, DTYPE, DEVICE,
)

In [ ]:
RESULTS_DIR = Path("results/grid/uci_bnn")
DATASETS    = ["boston"]  # add "naval", "energy" once those runs exist
N_SPLITS    = 5

SAMPLER_LABELS = {
    "grid_boomerang":        "Grid Boomerang",
    "grid_sticky_boomerang": "Grid Sticky Boomerang",
    "nuts":                  "NUTS",
    "nuts_horseshoe":        "NUTS-HS",
}

## Prediction helpers

Unlike the old tree's `TorchTarget`/`ModuleGaussianLikelihood.predict()`, `BayesianModule` exposes `bm.module`/`bm.param_dict_fn` directly -- predictions go through `torch.func.functional_call`, same pattern as `grid_toy_results.ipynb`. Every run here has a trailing `log_sigma` coordinate (noise is always learned), so `predict_from_samples` doesn't need a learned/fixed branch.

In [ ]:
@torch.no_grad()
def predict_from_samples(samples: Tensor, bm, X_test: Tensor) -> tuple[Tensor, Tensor, float]:
    """Returns (mean_pred, epistemic_std, noise_std).

    noise_std is the posterior mean of sigma = exp(log_sigma) -- every
    sampler in this pipeline learns the noise, so samples always carries
    a trailing log_sigma column.
    """
    weight_samples = samples[:, :-1]
    noise_std = float(samples[:, -1].exp().mean())

    preds = torch.stack([
        torch.func.functional_call(bm.module, bm.param_dict_fn(beta), (X_test,)).squeeze(-1)
        for beta in weight_samples
    ])  # [S, N]
    mean_pred = preds.mean(0)
    epist_std = preds.std(0)

    return mean_pred, epist_std, noise_std

## Metrics

In [ ]:
def rmse(y_true: Tensor, mean_pred: Tensor, y_std: float) -> float:
    return float(((mean_pred - y_true) ** 2).mean().sqrt()) * y_std


def nll(y_true: Tensor, mean_pred: Tensor, epist_std: Tensor,
        noise_std: float, y_std: float) -> float:
    total_std = (epist_std ** 2 + noise_std ** 2).sqrt()
    ll = (
        -0.5 * ((y_true - mean_pred) / total_std) ** 2
        - total_std.log()
        - 0.5 * math.log(2 * math.pi)
    ).mean()
    return float(-ll + math.log(y_std))


def crps_gaussian(y_true: Tensor, mean_pred: Tensor, epist_std: Tensor,
                   noise_std: float, y_std: float) -> float:
    """Closed-form CRPS for a Gaussian predictive, on the original scale."""
    from torch.distributions import Normal
    sigma = (epist_std ** 2 + noise_std ** 2).sqrt() * y_std
    mu    = mean_pred * y_std
    yt    = y_true * y_std
    d     = Normal(0.0, 1.0)
    z     = (yt - mu) / sigma
    crps  = sigma * (z * (2 * d.cdf(z) - 1) + 2 * d.log_prob(z).exp() - 1 / math.sqrt(math.pi))
    return float(crps.mean())


def compute_metrics(y_true, mean_pred, epist_std, noise_std, y_std) -> dict:
    return {
        "RMSE": rmse(y_true, mean_pred, y_std),
        "NLL":  nll(y_true, mean_pred, epist_std, noise_std, y_std),
        "CRPS": crps_gaussian(y_true, mean_pred, epist_std, noise_std, y_std),
    }

## Load all runs and compute metrics

In [ ]:
print("Loading raw datasets for test-set reconstruction...")
raw = load_raw_datasets(tuple(DATASETS))
print("Done.", list(raw.keys()))

In [ ]:
records = []  # list of dicts: {dataset, sampler, split_id, RMSE, NLL, CRPS, wall time}

for dataset in DATASETS:
    ds_dir = RESULTS_DIR / dataset
    if not ds_dir.exists():
        print(f"  [{dataset}] no results directory found, skipping.")
        continue
    if dataset not in raw:
        print(f"  [{dataset}] not available in raw data, skipping.")
        continue

    X_all, y_all = raw[dataset]

    for split_id in range(N_SPLITS):
        split_dir = ds_dir / f"split_{split_id:02d}"
        if not split_dir.exists():
            continue

        data = make_split(X_all, y_all, seed=BASE_SEED + split_id, dtype=DTYPE, device=DEVICE)
        X_test = data["X_test"]
        y_test = data["y_test"]
        y_std  = data["y_std"]

        # Rebuild the target ONCE per split (all runs in a split share the
        # same architecture/config -- see uci_bnn_grid.py's run_split).
        bm = None

        for pt_path in sorted(split_dir.glob("*.pt")):
            run = torch.load(pt_path, map_location="cpu", weights_only=False)
            sampler = run["sampler"]
            wall_time = run["elapsed_sec"]

            if bm is None:
                cfg = BNNConfig(
                    layer_sizes=run["layer_sizes"],
                    activation=run["activation"],
                    prior_sigma_scale=run["prior_sigma_scale"],
                )
                bm, _, _ = build_target(data, cfg)

            samples = run["samples"].to(dtype=DTYPE)  # [S, D]

            try:
                mean_pred, epist_std, noise_std = predict_from_samples(samples, bm, X_test)
                m = compute_metrics(y_test, mean_pred, epist_std, noise_std, y_std)
                records.append({
                    "dataset":  dataset,
                    "sampler":  sampler,
                    "split_id": split_id,
                    **m,
                    "wall time": wall_time,
                })
                print(f"  [{dataset} / split {split_id} / {sampler}]  "
                      f"RMSE={m['RMSE']:.3f}  NLL={m['NLL']:.3f}  CRPS={m['CRPS']:.3f}")
            except Exception as e:
                print(f"  [{dataset} / split {split_id} / {sampler}] ERROR: {e}")

df = pd.DataFrame(records)
print(f"\nTotal records: {len(df)}")

## Aggregate: mean ± SEM across splits

In [ ]:
def mean_sem(x):
    return x.mean(), x.sem()

rows = []
for (dataset, sampler), g in df.groupby(["dataset", "sampler"]):
    row = {"Dataset": dataset, "Sampler": SAMPLER_LABELS.get(sampler, sampler)}
    for metric in ["RMSE", "NLL", "CRPS", "wall time"]:
        mu, sem = mean_sem(g[metric])
        if metric == "wall time":
            mu_min = mu / 60
            if sem >= 60:
                mu_min += sem // 60
                sem_min = sem % 60
                row[metric] = f"{mu_min:.1f}min ± {sem_min:.1f}s"
            else:
                row[metric] = f"{mu_min:.1f}min ± {sem:.1f}s"
        else:
            row[metric] = f"{mu:.5f} ± {sem:.3f}"
    rows.append(row)

table = pd.DataFrame(rows).set_index(["Dataset", "Sampler"])
table

## LaTeX export

In [ ]:
print(table.to_latex(escape=False))